<img src="http://imgur.com/1ZcRyrc.png" style="float: left; margin: 20px; height: 55px" />

# Joins Review!

_Author:_ Tim Book (Excel lab), Rowan S

# Our Mission
This exercise builds on the Excel lab, which you're already familiar with!

Pandas .merge() is [based on SQL joins](https://github.com/pandas-dev/pandas/issues/220). In the SQL unit you will learn different types of joins more in depth.

## Imports

In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore", category = UserWarning, module = "openpyxl")

Read in the Excel data from Jan 1.

In [2]:
jan1 = pd.read_excel('./data/Jan 1.xlsx', sheet_name='Boston')

We also have a `plu-codes.csv` file containing product names matched up against their PLU codes.

In [3]:
plu = pd.read_csv("./data/plu-codes.csv")
plu.head()

,product,plu_code
0,Apple (Fuji),4131
1,Apple (Gala),4134
2,Apricot,3302
3,Avocado,4225
4,Banana,4011


## Update Data

Someone forgot to include three rows in the Jan 1 sales spreadsheet. Let's add those.

In [4]:
lost_sales = pd.DataFrame([
    {'prodcode':84002, 'price_eu':1.8, 'weight_kg':4.4, 'quantity':170},
    {'prodcode':7007, 'price_eu':1.2, 'weight_kg':1.4, 'quantity':396},
    {'prodcode':1001, 'price_eu':4, 'weight_kg':1.1, 'quantity':2}
])

jan1 = pd.concat([jan1, lost_sales])

Some stores have decided to stock new <a href='https://banana-label-catalog.org/catalog/content'>fruits</a>, veggies, and convenience items. Add them to the plu_codes key.

In [5]:
new_products_dict = {'Apple (Honeycrisp)': 3131,
                'Starfruit': 3271,
                'Mango': 3002,
                'Bananas (Organic)': 94011,
                'Bananas, Hawaiian Plantain': 3287,
                'Brussels Sprouts': 3400,
                'Lima Beans': 3036,
                'Mini Donuts': 6472,
                'Windex': 1001,
                'Baking Soda': 1073,
                'Cat Litter': 2060}

new_products = pd.DataFrame({'product': list(new_products_dict.keys()), 'plu_code': list(new_products_dict.values())})

plu = pd.concat([plu, new_products], axis=0, ignore_index=True)

Take a look at the datasets:

In [6]:
jan1.head(5) # This is our "left" dataframe.

,prodcode,price_eu,weight_kg,quantity
0,4134,1.957576,7.857868,498
1,4012,0.912565,5.746554,384
2,4323,0.740753,8.748904,431
3,4078,1.571533,7.061995,457
4,4323,1.389404,1.117369,316


In [7]:
plu.head(5) # This is our "right" dataframe.

,product,plu_code
0,Apple (Fuji),4131
1,Apple (Gala),4134
2,Apricot,3302
3,Avocado,4225
4,Banana,4011


## pd.merge
In the original Excel lab, you merged the product names from the `plu` dataframe onto the `jan1` dataframe.

<code>Jan1</code> data doesn't have product names, only codes. The <code>plu</code> dataset contains a key of product codes and corresponding names.



You're used to doing this:

```jan1.merge(plu)```

Sometimes you specify inner or left or right. But what does it mean?

## Inner Join

An inner join returns all records that have a match in BOTH tables.

<img src="./imgs/inner.png" width=600><br />
<sup><a href="https://www.programiz.com/sql/inner-join#google_vignette">source</a></sup>

In [9]:
inner = pd.merge(left = jan1, right = plu, how = 'inner')
inner.head(5)

MergeError: No common columns to perform merge on. Merge options: left_on=None, right_on=None, left_index=False, right_index=False

**An error!** What columns do we want to use as our "join" columns? They have two different column names. We need to set `left_on` and `right_on`.

In [10]:
inner = pd.merge(left = jan1, right = plu, left_on = 'prodcode', right_on = 'plu_code', how = 'inner')

In [11]:
# Check number of unique product codes
inner['prodcode'].nunique()

21

In [12]:
jan1['prodcode'].nunique()

23

In [13]:
# Check len()
len(inner), len(jan1)

(178, 180)

In [14]:
# Did this do what we wanted it to?


## Left Join

- A left join returns all records from the LEFT table, whether they have a match in the RIGHT table or not.
- Each record from the left table will be returned as many times as there are matching records on the right table.

<img src="./imgs/left.png" width=600><br />
<sup><a href="https://www.programiz.com/sql/inner-join#google_vignette">source</a></sup>

In [15]:
left = pd.merge(left = jan1, right = plu, how = 'left', left_on = 'prodcode', right_on = 'plu_code')
left.head(5)

,prodcode,price_eu,weight_kg,quantity,product,plu_code
0,4134,1.957576,7.857868,498,Apple (Gala),4134.0
1,4012,0.912565,5.746554,384,Orange,4012.0
2,4323,0.740753,8.748904,431,Strawberries,4323.0
3,4078,1.571533,7.061995,457,Corn,4078.0
4,4323,1.389404,1.117369,316,Strawberries,4323.0


In [16]:
# Check sizes. New DF should be the same size as Jan1. 
len(left), len(jan1)

(180, 180)

In [17]:
# Q: Sometimes it can be bigger. (Why?)

# A: Left join includes all matching entries from the right dataframe. If a value appears multiple times, all its occurrences will be included in the result.

In [18]:
# Did this do what we wanted it to?


## Right Join

A right join is the exact same as a left join but backwards. It returns all records from the RIGHT table, whether there's a match in the left table or not.

Right joins have specific use cases. You could use it interchangeably with left join, but you <a href='https://stackoverflow.com/questions/436345/when-or-why-would-you-use-a-right-outer-join-instead-of-left/41075976'>probably shouldn't.</a> Switching back and forth indiscriminately is confusing, and it's easier to make mistakes:

<img src="./imgs/right.png" width=600><br />

In [19]:
right = pd.merge(left = jan1, right = plu, how = 'right', left_on='prodcode', right_on='plu_code')
right.head(5)

,prodcode,price_eu,weight_kg,quantity,product,plu_code
0,4131.0,2.612334,7.238790,322.0,Apple (Fuji),4131
1,4131.0,1.336470,9.289587,215.0,Apple (Fuji),4131
2,4131.0,1.802106,3.334840,385.0,Apple (Fuji),4131
3,4131.0,2.510641,6.186207,423.0,Apple (Fuji),4131
4,4131.0,2.972409,3.824583,131.0,Apple (Fuji),4131


In [20]:
# Check sizes. New DF should be the same size as Jan1. 
len(right), len(jan1)

(188, 180)

In [21]:
# Did this do what we wanted it to?


## Full Outer Join

Outer joins return matched values and unmatched values from EITHER or BOTH tables.

<img src="./imgs/full_outer.png" width=500><br />

In [22]:
outer = pd.merge(left = jan1, right = plu, how = 'outer', left_on = 'prodcode', right_on = 'plu_code')
outer.head(5)

,prodcode,price_eu,weight_kg,quantity,product,plu_code
0,4134.0,1.957576,7.857868,498.0,Apple (Gala),4134.0
1,4134.0,2.987341,7.832501,452.0,Apple (Gala),4134.0
2,4134.0,2.679997,9.313293,112.0,Apple (Gala),4134.0
3,4134.0,2.207526,2.456283,388.0,Apple (Gala),4134.0
4,4134.0,2.532783,4.737317,123.0,Apple (Gala),4134.0


In [23]:
# Length of dataframes
len(outer), len(jan1)

(190, 180)

In [24]:
# Check nulls
outer.isna().sum()

prodcode     10
price_eu     10
weight_kg    10
quantity     10
product       2
plu_code      2
dtype: int64

In [25]:
outer[outer.isna().any(axis=1)] #oh no


,prodcode,price_eu,weight_kg,quantity,product,plu_code
177,84002.0,1.8,4.4,170.0,NaN,NaN
178,7007.0,1.2,1.4,396.0,NaN,NaN
180,NaN,NaN,NaN,NaN,Apple (Honeycrisp),3131.0
181,NaN,NaN,NaN,NaN,Starfruit,3271.0
182,NaN,NaN,NaN,NaN,Mango,3002.0
183,NaN,NaN,NaN,NaN,Bananas (Organic),94011.0
184,NaN,NaN,NaN,NaN,"Bananas, Hawaiian Plantain",3287.0
185,NaN,NaN,NaN,NaN,Brussels Sprouts,3400.0
186,NaN,NaN,NaN,NaN,Lima Beans,3036.0
187,NaN,NaN,NaN,NaN,Mini Donuts,6472.0


In [26]:
# What happened?:


In [27]:
# Did this do what we wanted it to?:


In [28]:
# Use cases for outer joins:
# Checking data quality, if your data is incorrect, cases with missing data.

## Cross Join

Cross joins return the cartesian product of sets of records from 2 or more tables. This generates all possible combinations of records.

<br /><img src="./imgs/shirts_pants.png" width=600><br />

In [29]:
cross = pd.merge(left = jan1, right = plu, how = 'cross')
cross.head(5)

,prodcode,price_eu,weight_kg,quantity,product,plu_code
0,4134,1.957576,7.857868,498,Apple (Fuji),4131
1,4134,1.957576,7.857868,498,Apple (Gala),4134
2,4134,1.957576,7.857868,498,Apricot,3302
3,4134,1.957576,7.857868,498,Avocado,4225
4,4134,1.957576,7.857868,498,Banana,4011


In [30]:
# Length of dataframes
len(cross), len(jan1)

(5580, 180)

We combined each row of one table, with each row of the other table.

In [31]:
len(jan1)*len(plu) 

5580

In [32]:
# Check nulls
cross.isna().sum()

prodcode     0
price_eu     0
weight_kg    0
quantity     0
product      0
plu_code     0
dtype: int64

In [33]:
# Does this make sense to do for this problem?:


In [34]:
# Use cases for cross joins:
# Use with caution and care...

### More resources

- <a href='https://pandas.pydata.org/docs/dev/user_guide/merging.html'>Pandas docs have good visual representations of joins!</a>
- <a href='https://nadiaslnv.medium.com/sql-101-choose-your-join-wisely-f5f34f8251b3'>Comparing joins</a>
- <a href='https://wizardzines.com/comics/joins/'>Julia Evans: rules for simple joins</a>